# Tutorial 1 — Generating a synthetic data set

This notebook builds a data set for a virtual MOLONARI monitoring point whose properties are **known**:

1. the boundary conditions are generated: river temperature, temperature at the bottom of the column, head difference;
2. the direct model of `pyheatmy` computes the temperatures at the three sensors inside the column;
3. measurement noise is added to these temperatures;
4. the data set is written in `.data/<point name>/`, in the same format as the processed data of a real point (`Point034`), together with a `truth.json` file holding the parameters used.

Tutorial 2 (`02_inversion.ipynb`) reads only these files, estimates the parameters of the river bed by MCMC and compares them with `truth.json`.

The notebook is deterministic: the noise comes from a seeded generator, so running it again writes the same files. The files depend on the direct model, so they must be generated again whenever `pyheatmy` changes.

In [ ]:
import json
import tempfile
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pyheatmy import Column, Layer, synthetic_MOLONARI
from pyheatmy.config import ZERO_CELSIUS

## 1. Settings

All the choices of the data set are made here. Temperatures are given in °C and converted to kelvins, the unit used inside `pyheatmy`.

- **Time.** Two weeks sampled every 15 minutes, as in the real data of `Point034`.
- **Column.** Four sensors at 0.1, 0.2, 0.3 and 0.4 m below the river bed. The deepest one gives the bottom boundary condition; the three others are the measurements the inversion has to reproduce.
- **Boundary conditions.** A daily sinusoid for the river temperature, a constant temperature at the bottom, and a constant head difference $\Delta H = H(0) - H(z_\text{bottom})$.
- **River bed.** One homogeneous layer. The head difference is chosen so that advection and conduction have the same order of magnitude (Péclet number close to 1), so that the temperatures carry information on both the permeability and the thermal properties.
- **Noise.** Gaussian, with the standard deviation usually assumed for the temperature sensors.

In [ ]:
POINT_NAME = "Synthetic01"
SEED = 20261026

# Time
START = (2011, 8, 1)
END = (2011, 8, 15)                 # excluded
TIME_STEP_S = 15 * 60

# Column: depths in m below the river bed; the last sensor gives the bottom boundary condition
DEPTH_SENSORS = [0.1, 0.2, 0.3, 0.4]
OFFSET = 0.0
NB_CELLS = 100

# Boundary conditions
RIVER_T_MEAN_C = 20.0
RIVER_T_AMPLITUDE_C = 5.0
RIVER_T_PERIOD_S = 24 * 3600
RIVER_T_PHASE = 0.0
BOTTOM_T_C = 13.0
HEAD_DIFFERENCE_M = 0.05

# River bed: one homogeneous layer, reaching the deepest sensor
LAYER = {
    "name": "sand",
    "zLow": DEPTH_SENSORS[-1],
    "IntrinK": 1e-12,     # intrinsic permeability, m2
    "n": 0.1,             # porosity
    "lambda_s": 2.0,      # thermal conductivity of the solid, W/m/K
    "rhos_cs": 4e6,       # volumetric heat capacity of the solid, J/m3/K
    "q_s": 0.0,           # lateral water source, 1/s
}

# Measurement noise on the temperatures of the three inner sensors, K
SIGMA_T = 0.1

DATA_DIR = Path(".data") / POINT_NAME

## 2. Boundary conditions

`synthetic_MOLONARI` builds the time axis and the three boundary signals. Each signal is described as `[[offset], [amplitude, period, phase], ...]`; a signal with an offset only is constant.

No noise is added at this stage (`sigma_meas_P = sigma_meas_T = None`): the boundary conditions written in the data set are exactly the ones the direct model is forced with, so that the inversion works with the true forcing.

In [ ]:
synthetic = synthetic_MOLONARI(
    offset=OFFSET,
    depth_sensors=DEPTH_SENSORS,
    param_time_dates=[START, END, TIME_STEP_S],
    param_dH_signal=[[HEAD_DIFFERENCE_M]],
    param_T_riv_signal=[[RIVER_T_MEAN_C + ZERO_CELSIUS], [RIVER_T_AMPLITUDE_C, RIVER_T_PERIOD_S, RIVER_T_PHASE]],
    param_T_aq_signal=[[BOTTOM_T_C + ZERO_CELSIUS]],
    sigma_meas_P=None,
    sigma_meas_T=None,
    verbose=False,
)

dates = list(synthetic._dates)
print(f"{len(dates)} time steps, from {dates[0]} to {dates[-1]}")

fig, (ax_t, ax_h) = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
ax_t.plot(dates, synthetic._T_riv - ZERO_CELSIUS, label="river")
ax_t.plot(dates, synthetic._T_aq - ZERO_CELSIUS, label=f"bottom ({DEPTH_SENSORS[-1]} m)")
ax_t.set_ylabel("T (°C)")
ax_t.legend()
ax_h.plot(dates, synthetic._dH)
ax_h.set_ylabel(r"$\Delta H$ (m)")
fig.tight_layout()

## 3. Direct model

A `Column` is forced with these boundary conditions and solved with the parameters of `LAYER`.

The `Column` takes two lists of dated measurements: `dH_measures`, with the head difference and the river temperature, and `T_measures`, with the four sensor temperatures. For the direct model, only the last sensor (the bottom boundary condition) and the initial profile are used; the values given for the three inner sensors are a linear interpolation between the river and the bottom, which is also the initial temperature profile.

The `Column` writes its own output files in a temporary folder (`rac`), not in the home directory.

In [ ]:
column = Column(
    river_bed=DEPTH_SENSORS[-1],
    depth_sensors=DEPTH_SENSORS,
    offset=OFFSET,
    dH_measures=synthetic._molonariP_data,
    T_measures=synthetic._T_Shaft_measures,
    nb_cells=NB_CELLS,
    rac=tempfile.mkdtemp(),
)
column.set_layers(Layer.from_dict(LAYER))
column.compute_solve_transi(verbose=False)

# Rows: river, the three inner sensors, bottom; temperatures in K
temperatures = column.get_temperature_at_sensors()
exact_inner = temperatures[1:-1]

darcy_velocity = column.get_flows_solve()
print(f"Darcy velocity: mean {darcy_velocity.mean():.2e} m/s, from {darcy_velocity.min():.2e} to {darcy_velocity.max():.2e} m/s")

## 4. Measurement noise

Gaussian noise of standard deviation `SIGMA_T` is added to the three inner sensors only. The generator is seeded, so the same noise is drawn at every run.

In [ ]:
rng = np.random.default_rng(SEED)
measured_inner = exact_inner + rng.normal(0.0, SIGMA_T, size=exact_inner.shape)

fig, ax = plt.subplots(figsize=(10, 4))
for depth, exact, measured in zip(DEPTH_SENSORS[:-1], exact_inner, measured_inner):
    line, = ax.plot(dates, exact - ZERO_CELSIUS, label=f"{depth} m, model")
    ax.plot(dates, measured - ZERO_CELSIUS, ".", markersize=2, color=line.get_color(), alpha=0.4)
ax.set_ylabel("T (°C)")
ax.set_title("Temperatures at the inner sensors: model (lines) and noisy measurements (dots)")
ax.legend()
fig.tight_layout()

## 5. Writing the data set

The files follow the processed data of `Point034` (`data/Point034_processed/` in MOLONARI-computo), so that the same reader works for a real and a virtual point:

| File | Columns |
|---|---|
| `processed_pressures.csv` | date, head difference (m), river temperature (°C) |
| `processed_temperatures.csv` | date, temperatures of the four sensors (°C) |
| `<point name>_info.csv` | description of the point, one `key,value` per line |
| `truth.json` | every setting of this notebook, including the layer parameters to be found by the inversion |

Dates are written as `yy/mm/dd HH:MM`. The head difference is written in metres, as the column header says: no sensor calibration has to be applied when reading it.

In [ ]:
DATE_FORMAT = "%y/%m/%d %H:%M"
DATE_HEADER = "Date Heure, GMT+01:00"

DATA_DIR.mkdir(parents=True, exist_ok=True)
date_strings = [d.strftime(DATE_FORMAT) for d in dates]

pressures = pd.DataFrame({
    DATE_HEADER: date_strings,
    "Pression différentielle (m)": np.round(synthetic._dH.astype(float), 6),
    "Temperature (°C)": np.round(temperatures[0] - ZERO_CELSIUS, 4),
})
pressures.to_csv(DATA_DIR / "processed_pressures.csv", index=False)

sensor_temperatures = np.vstack([measured_inner, temperatures[-1:]]) - ZERO_CELSIUS
thermal = pd.DataFrame({DATE_HEADER: date_strings})
for i, row in enumerate(sensor_temperatures, start=1):
    thermal[f"Température {i} (°C)"] = np.round(row, 4)
thermal.to_csv(DATA_DIR / "processed_temperatures.csv", index=False)

info = {
    "Point_Name": POINT_NAME,
    "P_Sensor_Name": "virtual",
    "Shaft_Name": "virtual",
    "Implantation_Date": dates[0].strftime("%Y/%m/%d %H:%M:%S"),
    "Meas_Date": dates[-1].strftime("%Y/%m/%d %H:%M:%S"),
    "River_Bed": DEPTH_SENSORS[-1],
    "Delta_h": OFFSET,
}
pd.Series(info).to_csv(DATA_DIR / f"{POINT_NAME}_info.csv", header=False)

truth = {
    "point_name": POINT_NAME,
    "seed": SEED,
    "time": {"start": list(START), "end_excluded": list(END), "time_step_s": TIME_STEP_S},
    "column": {"depth_sensors_m": DEPTH_SENSORS, "offset_m": OFFSET, "nb_cells": NB_CELLS},
    "boundary_conditions": {
        "river_temperature_mean_C": RIVER_T_MEAN_C,
        "river_temperature_amplitude_C": RIVER_T_AMPLITUDE_C,
        "river_temperature_period_s": RIVER_T_PERIOD_S,
        "river_temperature_phase_rad": RIVER_T_PHASE,
        "bottom_temperature_C": BOTTOM_T_C,
        "head_difference_m": HEAD_DIFFERENCE_M,
    },
    "layers": [LAYER],
    "noise": {"sigma_T_K": SIGMA_T, "applied_to": "the three inner sensors"},
}
(DATA_DIR / "truth.json").write_text(json.dumps(truth, indent=2) + "\n")

for path in sorted(DATA_DIR.iterdir()):
    print(path)

## 6. Check: reading the files back

The files are read as tutorial 2 will read them, and compared with the arrays above. The only differences allowed are the rounding of the written values.

In [ ]:
read_pressures = pd.read_csv(DATA_DIR / "processed_pressures.csv")
read_thermal = pd.read_csv(DATA_DIR / "processed_temperatures.csv")
read_dates = pd.to_datetime(read_thermal.iloc[:, 0], format=DATE_FORMAT)

TEMPERATURE_ROUNDING = 5e-5     # half of the last written decimal, K
HEAD_ROUNDING = 5e-7            # m

assert list(read_dates) == dates
assert (read_pressures.iloc[:, 0] == read_thermal.iloc[:, 0]).all()
assert np.allclose(read_pressures.iloc[:, 1], synthetic._dH, rtol=0, atol=HEAD_ROUNDING)
assert np.allclose(read_pressures.iloc[:, 2] + ZERO_CELSIUS, temperatures[0], rtol=0, atol=TEMPERATURE_ROUNDING)
assert np.allclose(read_thermal.iloc[:, 1:].to_numpy().T + ZERO_CELSIUS,
                   np.vstack([measured_inner, temperatures[-1:]]), rtol=0, atol=TEMPERATURE_ROUNDING)
print("The files hold the generated data set.")